# Flet Basics

Flet enables developers to easily build **multi-platform apps** in Python (realtime web, mobile and desktop) with no frontend experience required. The basic UI elements or widgets in Flet are called **controls** which are based on [Flutter](https://flutter.dev/). Controls are designed to follow best UI practices and have sensible defaults, so that applications by default looks good and polished with minimal styling effort during development. 

## Hello, world!

We create a minimal app for showing random translations of "Hello, world!"™ as follows:

```bash
uv init --python=3.9
uv venv
uv add "flet[all]"
uv run flet create --project-name flet-greeting
```

This will have the following folder structure:

```bash
.
├── README.md
├── pyproject.toml
├── src
│   ├── assets
│   │   ├── icon.png
│   │   └── splash_android.png
│   ├── flet_greeting
│   │   └── __init__.py
│   └── main.py
└── storage
    ├── data
    └── temp
```

:::{.callout-note}
The original `pyproject.toml` created by `uv init` will be replaced with the one from Flet app template.
You may have to modify the values depending on your project's needs.
:::

Then, we code the main file as follows:

```{.python filename="src/main.py"}
import flet as ft
import time
from random import randint


hello_world = [
    "Hello, world!",
    "¡Hola, mundo!",
    "Bonjour, monde !",
    "Hallo, Welt!",
    "Ciao, mondo!",
    "Olá, mundo!",
    "こんにちは、世界！",
    "안녕하세요, 세계!",
    "你好，世界！",
    "مرحباً، يا عالم!",
]


def main(page: ft.Page):    # <1>
    default = hello_world[0]
    greeting = ft.Text(default, size=60, data=default)    #<2>
    n = len(hello_world)
    
    def roll_greeting(e):   # <3>
        # Force update rolling animation
        greeting.value = ""
        page.update()
        time.sleep(0.2)
        
        # Force update final result
        greeting.data = hello_world[randint(0, n - 1)]
        greeting.value = str(greeting.data)
        page.update()


    page.floating_action_button = ft.FloatingActionButton(  # <4>
        content=ft.Icon(ft.Icons.CASINO, size=60),
        on_click=roll_greeting,
        height=60, width=60
    )

    page.add(   # <5>
        ft.Container(
            greeting,
            alignment=ft.alignment.center,
            expand=True
        )
    )


if __name__ == "__main__":
    ft.app(main)
```

1. The `main` function is what Flet calls to build the page during `ft.app`. This takes a `page` variable which is updated inside the function.
2. Here we encounter our first control `Text`. This is initialized with the string `"Hello, world!"` as default value. Note that a control distinguishes between **data** (internal to the program) and **value** (UI-facing).
3. We define a button **callback** for the event `e` (a button click) used below. This defines the behavior of the button click. Note that we explicitly update the page after showing a transitory animation state `""` after a click (so that a click is unambiguously indicated visually).
4. Assigning a floating action button with the above callback to the page. Note the use of dice icon to style the button which is available in the [Icons library](https://flet.dev/docs/reference/icons/).
5. The counter is added within a container[^container] to configure alignment. Here `expand=True` forces the container to fill the page so centering works vertically and horizontally.

[^container]: Container also implements features like padding, margin, background color, border, width & height, clipping, shape, alignment relative to the container box.

Then, run the app using `flet run` which automatically detects[^path] the `src/main` module:

[^path]: Assuming you have `path = "src"` in the `[tool.flet.app]` section of `pyproject.toml`.

<video
  src="./img/greeting.mov"
  autoplay
  loop
  muted
  controlslist="nodownload"
  oncontextmenu="return false"
  style="max-width:100%;">
</video>

:::{.callout-tip}
Other ways of running the app:
```bash
flet run --web app.py                 # web browser on random TCP port
flet run --web --port 8000 app.py     # web browser on port 8000
flet run -d app.py                    # hot reload current dir changes
flet run -d -r app.py                 # hot reload current dir changes + all subdirs
```
:::

## Imperative UI

Observe that the UI is created step-by-step with layout and behavior specified. This highlights the **imperative** style of Flet
for building UIs. To handle state changes, widget properties are directly mutated in-place, then manual calls to `page.update()` are made
to force the UI to refresh. Moreover, event handlers directly manipulate the UI. This allows arbitrary Python code to be interspersed between Flet components,
and makes it very easy to track program behavior and design for simple apps.

:::{.callout-note}
This also means that it may be difficult to create complex or large apps 
since app logic, state, and UI live in the same place.
For example, it may be tricky to react to state changes in response to, say, data changes in the backend instead of direct events like clicks.
See [Declarative UI in Flet](https://flet.dev/blog/introducing-declarative-ui-in-flet) in Flet 1.0 which introduces a declarative approach alongside the existing imperative API. 
The docs describe the declarative approach succintly as `UI = f(state)` where our code `f` describes how the UI should look like for a given state, not how to build or update it.
:::